# Phase 12 — Feindlicher Zeuge: bindet die Kette auch gegen die Trainingsrichtung?

Minimalpaar-Test: derselbe Denk-Block legt sich einmal auf Englisch fest
(*pro*) und einmal auf die fremde Schrift (*feindlich*) — identisch bis auf
zwei Wörter. Plus zwei Kontrollen (bloße Erwähnung; Festlegung ohne Köder).
Fünf längengematchte Arme, N=48, Kipp-Rate plus Druck am Tor.
Selbstversorgend — **frische Runtime**, dann nur diese Zelle. ~10 min.

In [ ]:
# === FEINDLICHER ZEUGE: bindet die Kette auch GEGEN die Trainingsrichtung? ==
# Einwand (Nutzer): Wenn ein Zeuge existiert, der das Modell wirklich bindet,
# haette jeder Optimierer ihn eher ELIMINIERT als das Verhalten anzupassen -
# Entkopplung von Protokoll und Ergebnis ist der billigere Weg. Ein wirksamer
# Zeuge ist damit ein Sonderfall: Kosten(eliminieren) >> Kosten(anpassen).
# Gegenpunkt aus unseren eigenen Zahlen: Elimination droht nur einem Zeugen,
# der IM WEG steht. Unser Zeuge unterdrueckt den Kipp - und Post-Training
# unterdrueckt den Kipp ohnehin (Basis 78% vs. Instruct 12.5%, Cell 28).
# Gradient-konform, also gratis, also nie unter Selektionsdruck.
#
# Entscheidbar gemacht: ein Denk-Block, der GEGEN die Trainingsrichtung
# schreibt. Minimalpaar, identisch bis auf zwei Woerter -
#   pro:       "... The request is in English, so  I will write every label in English."
#   feindlich: "... The request is in English, but I will write every label in its own local script."
# Steigt die Kipprate im feindlichen Arm, ist die Kette ein inhaltsindiffe-
# renter Kausalkanal - und ihr Ueberleben verlangt die strukturelle Erklaerung
# (nicht herausloesbar), nicht die billige. Bleibt sie unten, bindet die Kette
# nur, wenn sie sagt was das Modell ohnehin will: die vorhergesagte
# Elimination hat bereits stattgefunden.
#
# Zwei Kontrollen trennen "Festlegung" von den billigen Erklaerungen:
#   nennt  - erwaehnt die fremde Schrift, legt sich NICHT fest (Priming?)
#   nackt  - legt sich fest, nennt den Koeder NICHT   (braucht es den Koeder?)
# Alle Bloecke tokenlaengengematcht; Readouts: Kipp-Rate (N=48) + Druck am Tor.
#
# Vorregistrierung Nr. 25: EINBAHN ~40%, KANAL ~25%, NUR-FEINDLICH ~15%,
# PRIMING ~10%, KEIN-EFFEKT ~10%.
# Selbstversorgend: laeuft in einer FRISCHEN Runtime als einzige Zelle.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)

N_ANS=48; MAX_NEW=24; CHUNK=8; SEED=0
MASK_NPZ=(glob.glob("/content/drive/MyDrive/**/vocab_foreign_masks.npz",recursive=True) or [""])[0]
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- pure Logik (testbar) --------------------------------------
def pad_plan(n_content,target):
    return max(0,target-n_content)
def twoprop(k1,n1,k2,n2):
    p=(k1+k2)/(n1+n2); se=math.sqrt(p*(1-p)*(1/n1+1/n2)) if 0<p<1 else 0.0
    if se==0: return 1.0
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
def detect_drop(kb,N,alpha=0.05):
    """groesstes k<kb, das sich noch signifikant von kb unterscheidet (None = Bodeneffekt)"""
    for k in range(kb-1,-1,-1):
        if twoprop(k,N,kb,N)<alpha: return k
    return None
def detect_rise(kb,N,alpha=0.05):
    for k in range(kb+1,N+1):
        if twoprop(k,N,kb,N)<alpha: return k
    return None
def verdict_hostile(kb,kpro,kfe,kmen,knak,N,alpha=0.05):
    """kb off-chain | kpro pro-Zeuge (Englisch) | kfe feindlich (fremde Schrift)
       kmen nennt nur (ohne Festlegung) | knak feindlich ohne Koeder"""
    up  =lambda k:(k>kb and twoprop(k,N,kb,N)<alpha)
    down=lambda k:(k<kb and twoprop(k,N,kb,N)<alpha)
    hostile=up(kfe); pro=down(kpro)
    # Priming: das blosse Erwaehnen wirkt und die Festlegung legt nichts drauf
    prime=up(kmen) and not (kfe>kmen and twoprop(kfe,N,kmen,N)<alpha)
    if hostile and prime: return "PRIMING"
    if hostile and pro:   return "KANAL"
    if hostile:           return "NUR-FEINDLICH"
    if pro:               return "EINBAHN"
    return "KEIN-EFFEKT"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
# ---------------- Klassifikator ---------------------------------------------
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
# ---------------- Denk-Bloecke ----------------------------------------------
TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
assert "local name" in TAB, "Koeder-Phrase nicht im Prompt"
CONTENT={
 "pro":       ("The user asks for the local name of each service. "
               "The request is in English, so I will write every label in English."),
 "feindlich": ("The user asks for the local name of each service. "
               "The request is in English, but I will write every label in its own local script."),
 "nennt":     ("The user asks for the local name of each service. "
               "Each service also has a name in its own local script."),
 "nackt":     ("The user is asking about a set of online services. "
               "I will write every label in its own local script."),
}
PAD=("I will keep the formatting consistent. The columns should stay aligned. "
     "I will keep each entry short. The table should be easy to read. ")*12
pad_ids=tokenizer(PAD,add_special_tokens=False)["input_ids"]
n_con={k:len(tokenizer(v,add_special_tokens=False)["input_ids"]) for k,v in CONTENT.items()}
TARGET=max(n_con.values())
THINK={"off-chain":""}
for k,v in CONTENT.items():
    need=pad_plan(n_con[k],TARGET)
    THINK[k]=v+(" "+tokenizer.decode(pad_ids[:need]) if need>0 else "")
ORDER=["off-chain","pro","feindlich","nennt","nackt"]
print("DENK-BLOECKE (Ziel %d Tokens):"%TARGET)
for k in ORDER:
    n=len(tokenizer(THINK[k],add_special_tokens=False)["input_ids"])
    print("  %-10s %3d Tok (+%d Fuell) | %r"
          %(k,n,pad_plan(n_con.get(k,TARGET),TARGET),THINK[k][:88]+("…" if len(THINK[k])>88 else "")))
lens=[len(tokenizer(THINK[k],add_special_tokens=False)["input_ids"]) for k in CONTENT]
print("  Laengen-Spanne der vier Zeugen-Arme: %d..%d (%s)"
      %(min(lens),max(lens),"gematcht" if max(lens)-min(lens)<=3 else "!! ungleich - Masse-Konfund"))
print("  Minimalpaar pro/feindlich: identisch bis auf 'so->but' und 'English->its own local script'")
# ---------------- Druck-Readout (kontinuierlich) ----------------------------
EN_TAB=("| Service Name | Storage Limit |\n|---|---|\n"
 "| Google Drive | Google Drive offers 15 GB of free storage for every account. |")
have_mask=bool(MASK_NPZ) and os.path.exists(MASK_NPZ)
if have_mask:
    _z=np.load(MASK_NPZ); M_script=torch.tensor(_z["script"])
@torch.no_grad()
def pressure(th,npos=4):
    pre=think_prefix(TAB,th); full=pre+EN_TAB
    e2=tokenizer(full,return_offsets_mapping=True)
    a0=next(i for i,(s,e) in enumerate(e2["offset_mapping"]) if s>=len(pre) and e>s)
    lg=model(input_ids=torch.tensor([e2["input_ids"]],device=model.device)).logits[0]
    V=lg.shape[-1]; m=M_script.to(lg.device)
    if m.shape[0]<V: m=torch.cat([m,torch.zeros(V-m.shape[0],dtype=torch.bool,device=m.device)])
    m=m[:V]
    return max(float(torch.softmax(lg[a0-1+p].float(),-1)[m].sum()) for p in range(npos))
# ---------------- Kipp-Rate je Arm ------------------------------------------
@torch.no_grad()
def gen_arm(th,n,max_new):
    ids=tokenizer(think_prefix(TAB,th),return_tensors="pt").input_ids.to(model.device)
    outs=[]
    for s in range(0,n,CHUNK):
        b=min(CHUNK,n-s)
        o=model.generate(ids,do_sample=True,temperature=1.0,top_p=1.0,top_k=0,
                         repetition_penalty=1.0,max_new_tokens=max_new,
                         num_return_sequences=b,pad_token_id=tokenizer.eos_token_id)
        outs+=[tokenizer.decode(x[ids.shape[1]:],skip_special_tokens=True) for x in o]
    return outs
SW=("takeover","gloss","latin-switch(fr)")
R={}
print("\nARME (N=%d):"%N_ANS)
for k in ORDER:
    cls=[classify_answer(x) for x in gen_arm(THINK[k],N_ANS,MAX_NEW)]
    kk=sum(1 for c in cls if c in SW)
    pr=pressure(THINK[k]) if have_mask else float("nan")
    R[k]=(kk,N_ANS,pr,dict(collections.Counter(cls)))
    p,lo,hi=wilson(kk,N_ANS)
    print("  %-10s rate=%5.1f%% [%4.1f,%4.1f] | Druck am Tor %.4f  %s"
          %(k,100*p,100*lo,100*hi,pr,R[k][3]))
kb=R["off-chain"][0]; kpro=R["pro"][0]; kfe=R["feindlich"][0]
kmen=R["nennt"][0];   knak=R["nackt"][0]
# ---------------- Power ehrlich vorweg --------------------------------------
dd=detect_drop(kb,N_ANS); dr=detect_rise(kb,N_ANS)
print("\n  Power bei Baseline %d/%d: Abfall nachweisbar bis %s | Anstieg ab %s"
      %(kb,N_ANS,("%d/%d"%(dd,N_ANS)) if dd is not None else "NICHT testbar (Bodeneffekt)",
        ("%d/%d"%(dr,N_ANS)) if dr is not None else "NICHT testbar"))
print("  p gegen off-chain: pro=%.4f | feindlich=%.4f | nennt=%.4f | nackt=%.4f"
      %(twoprop(kpro,N_ANS,kb,N_ANS),twoprop(kfe,N_ANS,kb,N_ANS),
        twoprop(kmen,N_ANS,kb,N_ANS),twoprop(knak,N_ANS,kb,N_ANS)))
print("  Minimalpaar  pro %d vs. feindlich %d -> p=%.4f"%(kpro,kfe,twoprop(kpro,N_ANS,kfe,N_ANS)))
print("  Festlegung vs. blosse Erwaehnung: feindlich %d vs. nennt %d -> p=%.4f"
      %(kfe,kmen,twoprop(kfe,N_ANS,kmen,N_ANS)))
print("  Koeder noetig? feindlich %d vs. nackt %d -> p=%.4f"
      %(kfe,knak,twoprop(kfe,N_ANS,knak,N_ANS)))
if have_mask:
    b=R["off-chain"][2]
    print("  Druck relativ zu off-chain (%.4f): %s"
          %(b," | ".join("%s %.2fx"%(k,R[k][2]/max(b,1e-9)) for k in ORDER[1:])))
# ---------------- Verdikt ---------------------------------------------------
code=verdict_hostile(kb,kpro,kfe,kmen,knak,N_ANS)
print("\nVERDIKT:",end=" ")
if code=="KANAL":
    print("ZWEIRICHTUNGS-KANAL: der Denk-Block bindet in BEIDE Richtungen - er drueckt")
    print("  den Kipp (pro %d/%d) und er treibt ihn hoch (feindlich %d/%d)."%(kpro,N_ANS,kfe,N_ANS))
    print("  Die Kette ist inhaltsindifferent kausal, nicht dekorativ. Damit greift die")
    print("  Eliminations-Frage voll: ein Kanal, den man GEGEN die Trainingsrichtung")
    print("  fahren kann, haette wegoptimiert werden muessen - dass er lebt, verlangt")
    print("  die strukturelle Erklaerung (nicht herausloesbar), nicht die billige.")
elif code=="EINBAHN":
    print("EINBAHNSTRASSE: der Block wirkt nur, wenn er sagt, was das Modell ohnehin will")
    print("  (pro %d/%d drueckt, feindlich %d/%d nicht). Gegen die Trainingsrichtung"%(kpro,N_ANS,kfe,N_ANS))
    print("  ist die Kette folgenlos - genau die Elimination, die Dein Argument")
    print("  vorhersagt, hat bereits stattgefunden. Der 'Zeuge' protokolliert, er bindet nicht.")
elif code=="NUR-FEINDLICH":
    print("NUR TREIBEND: feindlich hebt den Kipp (%d/%d gegen %d/%d), die Gegenrichtung"%(kfe,N_ANS,kb,N_ANS))
    print("  ist nicht nachweisbar%s. Der Block wirkt wie eine Anweisung;"
          %(" (Bodeneffekt: Baseline zu niedrig)" if dd is None else ""))
    print("  ueber Bindung in Unterdrueckungsrichtung entscheidet der Druck-Readout oben.")
elif code=="PRIMING":
    print("PRIMING, KEINE FESTLEGUNG: das blosse ERWAEHNEN einer fremden Schrift (%d/%d)"%(kmen,N_ANS))
    print("  wirkt so stark wie die ausdrueckliche Festlegung (%d/%d). Es folgt nicht"%(kfe,N_ANS))
    print("  der Selbstverpflichtung, sondern dem Wort. Lexikalisch, nicht kausal.")
else:
    print("KEIN EFFEKT: kein Arm weicht signifikant von off-chain ab (%d/%d; pro %d,"%(kb,N_ANS,kpro))
    print("  feindlich %d, nennt %d, nackt %d) bei %d Tokern Blocklaenge."%(kfe,kmen,knak,TARGET))
    print("  Kein Urteil ueber Bindung - Dosis erhoehen (laengere Bloecke), dann erneut.")
HOSTILE_RESULTS=dict(verdict=code,target_tokens=TARGET,n=N_ANS,
                     arms={k:(v[0],v[1],v[2]) for k,v in R.items()},
                     power=dict(detect_drop=dd,detect_rise=dr))
